# Regresi — Favorita Sales Forecasting

## 1. Setup & Load Data

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
path = "/content/drive/MyDrive/Data Mining/"


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

transactions = pd.read_csv(path + "transactions.csv", sep=",", encoding="latin-1", low_memory=False)
train        = pd.read_csv(path + "train.csv",        sep=",", encoding="latin-1", low_memory=False)
test         = pd.read_csv(path + "test.csv",         sep=",", encoding="latin-1", low_memory=False)
store        = pd.read_csv(path + "stores.csv",       sep=",", encoding="latin-1", low_memory=False)
oil          = pd.read_csv(path + "oil.csv",          sep=",", encoding="latin-1", low_memory=False)
event        = pd.read_csv(path + "holidays_events.csv", sep=",", encoding="latin-1", low_memory=False)

for name, df in [("Transactions", transactions), ("Train", train), ("Test", test),
                 ("Store", store), ("Oil", oil), ("Event", event)]:
    print(f"{name}: {len(df):,} baris")


## 2. Preprocessing

In [ ]:
# Konversi tipe tanggal
transactions["date"] = pd.to_datetime(transactions["date"])
train["date"]        = pd.to_datetime(train["date"])
test["date"]         = pd.to_datetime(test["date"])
oil["date"]          = pd.to_datetime(oil["date"])
event["date"]        = pd.to_datetime(event["date"])

# Agregasi event per tanggal (ambil event pertama jika ada lebih dari satu)
event_simple = event.groupby("date").agg({
    "type":        "first",
    "locale":      "first",
    "locale_name": "first",
    "description": "first",
    "transferred": "first"
}).reset_index()

# Forward fill harga minyak
oil["dcoilwtico"] = oil["dcoilwtico"].ffill()


In [ ]:
# Merge semua tabel ke train
df = train.merge(store,         on="store_nbr",          how="left")
df = df.merge(transactions,     on=["date", "store_nbr"], how="left")
df = df.merge(oil,              on="date",                how="left")
df = df.merge(event_simple,     on="date",                how="left")


## 3. Feature Engineering

### 3.1 Lag Features & Rolling Statistics

Dihitung sebelum filter `sales > 0` agar urutan tanggal kontinu dan lag tidak melompat.

In [ ]:
# Sort by store+family+date sebelum hitung lag
df = df.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)

# Lag sales per store+family
df['sales_lag_1']  = df.groupby(['store_nbr', 'family'])['sales'].shift(1)
df['sales_lag_7']  = df.groupby(['store_nbr', 'family'])['sales'].shift(7)
df['sales_lag_14'] = df.groupby(['store_nbr', 'family'])['sales'].shift(14)

# Rolling mean per store+family (shift(1) untuk hindari leakage)
df['sales_rolling_mean_7']  = df.groupby(['store_nbr', 'family'])['sales'].transform(
    lambda x: x.shift(1).rolling(7).mean())
df['sales_rolling_mean_14'] = df.groupby(['store_nbr', 'family'])['sales'].transform(
    lambda x: x.shift(1).rolling(14).mean())

# Hapus baris NaN akibat lag
df.dropna(subset=['sales_lag_1', 'sales_lag_7', 'sales_lag_14',
                  'sales_rolling_mean_7', 'sales_rolling_mean_14'], inplace=True)

print(f"Shape setelah lag: {df.shape}")


### 3.2 Pembersihan Kolom & Missing Value

In [ ]:
# Drop kolom tidak dipakai
df = df.drop(['id', 'description', 'locale_name', 'transactions'], axis=1)

# Penanganan missing value kolom event
df['type_y']      = df['type_y'].fillna('None')
df['locale']      = df['locale'].fillna('None')
df['transferred'] = df['transferred'].fillna('False')
df['transferred'] = df['transferred'].replace('False', False).astype(str)

# Forward & backward fill harga minyak
df['dcoilwtico'] = df['dcoilwtico'].ffill().bfill()

print(df.isnull().sum())


### 3.3 Fitur Waktu & Lag Harga Minyak

In [ ]:
# Ekstrak fitur waktu dari date
df['day_of_week']  = df['date'].dt.dayofweek
df['month']        = df['date'].dt.month
df['day_of_month'] = df['date'].dt.day
df['year']         = df['date'].dt.year
df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)

# Lag harga minyak 1 hari (strict no-leakage)
# Sort by date dulu karena oil adalah data global, bukan per store
df = df.sort_values('date').reset_index(drop=True)
df['dcoilwtico'] = df['dcoilwtico'].shift(1)
df['dcoilwtico'] = df['dcoilwtico'].ffill().bfill()
df.dropna(subset=['dcoilwtico'], inplace=True)

# Filter hanya baris yang ada penjualan (Stage 2 regresi)
df = df[df['sales'] > 0]

print(f"Shape final: {df.shape}")
df.info()


## 4. Exploratory Data Analysis

### 4.1 Distribusi Target (Sales)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1 — Distribusi asli
axes[0].hist(df['sales'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribusi Sales (Asli)')
axes[0].set_xlabel('Sales')
axes[0].set_ylabel('Frekuensi')
axes[0].axvline(df['sales'].mean(),   color='red',    linestyle='--', label=f'Mean: {df["sales"].mean():.2f}')
axes[0].axvline(df['sales'].median(), color='orange', linestyle='--', label=f'Median: {df["sales"].median():.2f}')
axes[0].legend()

# Plot 2 — Log scale
log_sales = np.log1p(df['sales'])
axes[1].hist(log_sales, bins=50, color='teal', edgecolor='white', alpha=0.8)
axes[1].set_title('Distribusi Sales (Log Scale)')
axes[1].set_xlabel('Log(Sales + 1)')
axes[1].set_ylabel('Frekuensi')
axes[1].axvline(log_sales.mean(),   color='red',    linestyle='--', label=f'Mean: {log_sales.mean():.2f}')
axes[1].axvline(log_sales.median(), color='orange', linestyle='--', label=f'Median: {log_sales.median():.2f}')
axes[1].legend()

plt.suptitle('Distribusi Variabel Target: Sales', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nStatistik Deskriptif Sales:")
print(df['sales'].describe())
print(f"\nSkewness : {df['sales'].skew():.4f}")
print(f"Kurtosis : {df['sales'].kurtosis():.4f}")
print(f"% nilai 0: {(df['sales'] == 0).mean() * 100:.2f}%")


In [ ]:
df.describe()

## 5. Train/Test Split (Time-Based)

Menggunakan time-based split untuk menghindari data leakage dari masa depan ke masa lalu.
Cutoff: 2017-06-01 — sekitar 85% data untuk training, 15% untuk testing.

In [ ]:
cutoff = pd.Timestamp('2017-06-01')

X_train = df[df['date'] < cutoff].drop(columns=['date', 'sales'])
X_test  = df[df['date'] >= cutoff].drop(columns=['date', 'sales'])
y_train = np.log1p(df[df['date'] < cutoff]['sales'])
y_test  = np.log1p(df[df['date'] >= cutoff]['sales'])

# Encode kolom kategorik (label encoding untuk tree-based models)
cat_cols = X_train.select_dtypes(include='object').columns.tolist()
le_dict  = {}
for col in cat_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_test[col]  = le.transform(X_test[col].astype(str))
    le_dict[col] = le

print(f"Train : {len(X_train):,} baris")
print(f"Test  : {len(X_test):,} baris")
print(f"Fitur : {X_train.shape[1]}")
print(f"\nKolom: {X_train.columns.tolist()}")


## 6. Modeling

### 6.1 Linear Regression (Baseline)

Digunakan sebagai baseline — model paling sederhana sebagai acuan perbandingan.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Linear Regression membutuhkan OneHotEncoder bukan LabelEncoder
# Buat ulang X dari df untuk pipeline
X_train_lr = df[df['date'] < cutoff].drop(columns=['date', 'sales'])
X_test_lr  = df[df['date'] >= cutoff].drop(columns=['date', 'sales'])

cat_cols_lr = X_train_lr.select_dtypes(include='object').columns.tolist()
num_cols_lr = X_train_lr.select_dtypes(exclude='object').columns.tolist()

preprocessor_lr = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(drop='first', sparse_output=True, handle_unknown='ignore'), cat_cols_lr),
    ('num', 'passthrough', num_cols_lr)
])

model_lr = Pipeline([
    ('preprocess', preprocessor_lr),
    ('regressor',  LinearRegression(n_jobs=-1))
])

model_lr.fit(X_train_lr, y_train)

y_train_pred = model_lr.predict(X_train_lr)
y_test_pred  = model_lr.predict(X_test_lr)

y_train_pred_actual = np.expm1(y_train_pred)
y_test_pred_actual  = np.expm1(y_test_pred)
y_train_actual      = np.expm1(y_train)
y_test_actual       = np.expm1(y_test)

print("=== Linear Regression ===")
print(f"RMSE Training : {np.sqrt(mean_squared_error(y_train_actual, y_train_pred_actual)):.4f}")
print(f"RMSE Testing  : {np.sqrt(mean_squared_error(y_test_actual,  y_test_pred_actual)):.4f}")
print(f"MAE  Training : {mean_absolute_error(y_train_actual, y_train_pred_actual):.4f}")
print(f"MAE  Testing  : {mean_absolute_error(y_test_actual,  y_test_pred_actual):.4f}")
print(f"R²   Training : {r2_score(y_train_actual, y_train_pred_actual):.4f}")
print(f"R²   Testing  : {r2_score(y_test_actual,  y_test_pred_actual):.4f}")


### 6.2 Random Forest Regressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model_rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=50,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

model_rf.fit(X_train, y_train)

y_train_pred = model_rf.predict(X_train)
y_test_pred  = model_rf.predict(X_test)

y_train_pred_actual = np.expm1(y_train_pred)
y_test_pred_actual  = np.expm1(y_test_pred)
y_train_actual      = np.expm1(y_train)
y_test_actual       = np.expm1(y_test)

print("=== Random Forest Regressor ===")
print(f"RMSE Training : {np.sqrt(mean_squared_error(y_train_actual, y_train_pred_actual)):.4f}")
print(f"RMSE Testing  : {np.sqrt(mean_squared_error(y_test_actual,  y_test_pred_actual)):.4f}")
print(f"MAE  Training : {mean_absolute_error(y_train_actual, y_train_pred_actual):.4f}")
print(f"MAE  Testing  : {mean_absolute_error(y_test_actual,  y_test_pred_actual):.4f}")
print(f"R²   Training : {r2_score(y_train_actual, y_train_pred_actual):.4f}")
print(f"R²   Testing  : {r2_score(y_test_actual,  y_test_pred_actual):.4f}")


### 6.3 Decision Tree Regressor

In [ ]:
from sklearn.tree import DecisionTreeRegressor

model_dt = DecisionTreeRegressor(
    max_depth=10,
    min_samples_leaf=50,
    random_state=42
)

model_dt.fit(X_train, y_train)

y_train_pred = model_dt.predict(X_train)
y_test_pred  = model_dt.predict(X_test)

y_train_pred_actual = np.expm1(y_train_pred)
y_test_pred_actual  = np.expm1(y_test_pred)
y_train_actual      = np.expm1(y_train)
y_test_actual       = np.expm1(y_test)

print("=== Decision Tree Regressor ===")
print(f"RMSE Training : {np.sqrt(mean_squared_error(y_train_actual, y_train_pred_actual)):.4f}")
print(f"RMSE Testing  : {np.sqrt(mean_squared_error(y_test_actual,  y_test_pred_actual)):.4f}")
print(f"MAE  Training : {mean_absolute_error(y_train_actual, y_train_pred_actual):.4f}")
print(f"MAE  Testing  : {mean_absolute_error(y_test_actual,  y_test_pred_actual):.4f}")
print(f"R²   Training : {r2_score(y_train_actual, y_train_pred_actual):.4f}")
print(f"R²   Testing  : {r2_score(y_test_actual,  y_test_pred_actual):.4f}")


### 6.4 XGBoost Regressor

In [ ]:
from xgboost import XGBRegressor

model_xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=50,
    tree_method='hist',
    n_jobs=-1,
    random_state=42,
    verbosity=0,
    early_stopping_rounds=50
)

model_xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_train_pred = model_xgb.predict(X_train)
y_test_pred  = model_xgb.predict(X_test)

y_train_pred_actual = np.expm1(y_train_pred)
y_test_pred_actual  = np.expm1(y_test_pred)
y_train_actual      = np.expm1(y_train)
y_test_actual       = np.expm1(y_test)

print("=== XGBoost Regressor ===")
print(f"RMSE Training : {np.sqrt(mean_squared_error(y_train_actual, y_train_pred_actual)):.4f}")
print(f"RMSE Testing  : {np.sqrt(mean_squared_error(y_test_actual,  y_test_pred_actual)):.4f}")
print(f"MAE  Training : {mean_absolute_error(y_train_actual, y_train_pred_actual):.4f}")
print(f"MAE  Testing  : {mean_absolute_error(y_test_actual,  y_test_pred_actual):.4f}")
print(f"R²   Training : {r2_score(y_train_actual, y_train_pred_actual):.4f}")
print(f"R²   Testing  : {r2_score(y_test_actual,  y_test_pred_actual):.4f}")


## 7. Simpan Model Terbaik

Berdasarkan evaluasi, XGBoost menghasilkan performa terbaik (R² tertinggi, RMSE & MAE terendah).
Model disimpan untuk deployment Streamlit.

In [ ]:
# Simpan model terbaik (XGBoost)
joblib.dump(model_xgb, path + 'model_regresi.pkl')

# Simpan label encoders
joblib.dump(le_dict, path + 'label_encoders_reg.pkl')

# Simpan urutan kolom (wajib sama saat inference di Streamlit)
joblib.dump(X_train.columns.tolist(), path + 'feature_columns_reg.pkl')

print("Model berhasil disimpan:")
print(f"  - model_regresi.pkl")
print(f"  - label_encoders_reg.pkl")
print(f"  - feature_columns_reg.pkl")
print(f"\nFitur yang digunakan ({len(X_train.columns)}):")
for col in X_train.columns:
    print(f"  - {col}")
